# Auth Log Analysis — SOC Triage Walkthrough

This notebook loads an authentication log CSV, performs initial triage (summary metrics, failed-login analysis), visualizes suspicious activity, performs a simple burst detection, and saves figures for demo/screenshots.

Run the dataset generator `data/generate_auth_logs.py` to create a larger dataset (default 1000 rows) or use `data/auth_logs_expanded_sample.csv` included here.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set(style='whitegrid')
DATA_PATH = Path('data/auth_logs_expanded_sample.csv')
DEMO_DIR = Path('demo')
DEMO_DIR.mkdir(exist_ok=True)


In [ ]:
# Load data
df = pd.read_csv(DATA_PATH, parse_dates=['timestamp'])
df.columns = [c.strip() for c in df.columns]
df.head()

## Basic summary metrics

In [ ]:
total = len(df)
by_action = df['action'].str.lower().value_counts()
print(f"Total events: {total}")
print('\nCounts by action:')
print(by_action.to_string())


## Failed-login analysis: counts by IP and user

In [ ]:
failed = df[df['action'].str.lower() == 'failed']
counts_ip = failed.groupby('src_ip').size().reset_index(name='failed_count').sort_values('failed_count', ascending=False)
counts_user = failed.groupby('user').size().reset_index(name='failed_count').sort_values('failed_count', ascending=False)
counts_ip.head(20)

## Plot: Top failed-login IPs

In [ ]:
top_n = 10
top_ips = counts_ip.head(top_n).set_index('src_ip')
plt.figure(figsize=(10,6))
sns.barplot(x='failed_count', y=top_ips.index, data=top_ips.reset_index(), palette='Reds_r')
plt.title(f'Top {top_n} IPs by failed logins')
plt.xlabel('Failed attempts')
plt.ylabel('Source IP')
plt.tight_layout()
plt.savefig(DEMO_DIR / 'top_failed_ips.png', dpi=150)
plt.show()


## Time-series: failed attempts over time (resampled)
Resample failed attempts by minute/hour to spot bursts.

In [ ]:
failed_ts = failed.set_index('timestamp').resample('5T').size().rename('failed_count')
plt.figure(figsize=(12,4))
failed_ts.plot()
plt.title('Failed authentication attempts (5-minute bins)')
plt.xlabel('Time')
plt.ylabel('Failed attempts')
plt.tight_layout()
plt.savefig(DEMO_DIR / 'failed_timeseries.png', dpi=150)
plt.show()


## Simple burst detection (sliding window per IP)
Identify IPs with more than N failures within M minutes (example: >5 failures within 10 minutes).

In [ ]:
def detect_bursts(df_failed, window_minutes=10, threshold=5):
    df_failed = df_failed.sort_values(['src_ip','timestamp'])
    suspect = []
    for ip, grp in df_failed.groupby('src_ip'):
        times = grp['timestamp'].sort_values().values
        # two-pointer sliding window
        i = 0
        for j in range(len(times)):
            while (times[j] - times[i]).astype('timedelta64[m]').astype(int) > window_minutes:
                i += 1
            if (j - i + 1) >= threshold:
                suspect.append({'src_ip': ip, 'count_in_window': j - i + 1, 'window_minutes': window_minutes})
                break
    return pd.DataFrame(suspect)

bursts = detect_bursts(failed, window_minutes=10, threshold=5)
bursts

## Triage notes & next steps
- Investigate top suspicious IPs: usernames attempted, timestamps, any successful logins nearby.
- Enrich IPs with GeoIP and ASN info (optional, see instructions).
- Check correlated telemetry (VPN, firewall, endpoint alerts).
- If confirmed malicious, block IP, reset affected accounts, and follow incident playbook.

Save the `demo/` images and include them as screenshots/GIFs in your portfolio README.